# Federated Learning for Privacy-Preserving Medical Diagnosis -- Kaggle Training

Real-data run of the **core architecture** on NIH ChestX-ray14 (GPU):
centralised ceiling + single-site floor + **federated FedAvg** global model.
Defence layers (DP, robust aggregation, attacks) are OFF -- this is the core
floor / federated / ceiling comparison (Tasks 4, 5, 6), trained properly
(class-balanced loss, compute-matched epochs).

## Before you run -- three settings (right sidebar)
1. **Accelerator -> GPU** (T4 x2 or P100).
2. **Internet -> On** (needed for `pip install` and `git clone`).
3. **Add Input** (the "+ Add Input" button) -> search and add:
   - `nih-chest-xrays/data`  (the official dataset -- gives the label CSVs; its images
     are full-res 1024px)
   - *(optional, faster)* `khanfashee/nih-chest-x-ray-14-224x224-resized` (224px images).
     If you add this, the notebook uses it automatically; if not, it falls back to the
     official full-res images.

Then Run All. Long run (~3-4 h at the default 30k images / 12 rounds). For an
unattended run use **Save Version -> Save & Run All (Commit)** so it finishes headless.


## 1. Check the GPU

In [ ]:
import torch, sys
print("Python:", sys.version.split()[0])
print("torch :", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Right sidebar -> Accelerator -> GPU, then Run All again.")

## 2. Install dependencies + clone the repo

torch / torchvision are preinstalled on Kaggle. We add Flower **core** (no Ray),
Opacus, and clone the project. Requires Internet = On (right sidebar).

Push your latest code to GitHub first. If the repo is private, change the URL to a
token form: `https://<TOKEN>@github.com/<user>/<repo>.git`.

In [ ]:
!pip install -q "flwr>=1.8,<2.0" "opacus>=1.4" torchmetrics scikit-learn pyyaml typer
import os
REPO_URL = "https://github.com/Pooja-Goel07/Federated-Learning-for-Privacy-Preserving-Medical-Diagnosis.git"
if not os.path.exists("/kaggle/working/fl"):
    !git clone -q $REPO_URL /kaggle/working/fl
%cd /kaggle/working/fl
!git log --oneline -1

## 3. Locate the mounted dataset

Kaggle mounts each added dataset read-only under `/kaggle/input/...`. This cell finds
the two label CSVs and the folder with the most PNG images (the resized mirror if you
added it, otherwise the official full-res images), with no downloading. It copies the
two small CSVs into a writable working folder and points the image root at the mount.

In [ ]:
import glob, os, shutil
NIH = "/kaggle/working/nih"
os.makedirs(NIH, exist_ok=True)

# label CSVs (search all mounted inputs)
de = glob.glob("/kaggle/input/**/Data_Entry_2017.csv", recursive=True)
bb = glob.glob("/kaggle/input/**/BBox_List_2017.csv", recursive=True)
assert de, "Data_Entry_2017.csv not found -- add the 'nih-chest-xrays/data' dataset as input"
shutil.copy(de[0], f"{NIH}/Data_Entry_2017.csv")
if bb:
    shutil.copy(bb[0], f"{NIH}/BBox_List_2017.csv")     # severity proxy (optional)

# image root = the mounted input dir containing the most PNGs
best, best_n = None, 0
for d in sorted(glob.glob("/kaggle/input/*")):
    n = sum(1 for _ in glob.iglob(d + "/**/*.png", recursive=True))
    print(f"  {d}: {n} png")
    if n > best_n:
        best, best_n = d, n
IMAGE_ROOT = best
print("CSVs ->", NIH, "| using Data_Entry:", de[0], "| BBox:", bb[0] if bb else "none")
print("IMAGE_ROOT:", IMAGE_ROOT, "| png count:", best_n)
assert best_n > 0, "no PNG images found in any mounted input"

## 4. Configure the run

Loads `configs/kaggle_real.yaml` and points it at the mounted data. Tweak
`max_images` / `rounds` to trade runtime for quality. Baseline epochs are kept equal to
the number of rounds so the centralised model is a **fair** ceiling.

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/fl")
from src.experiments.config import load_config

cfg = load_config("configs/kaggle_real.yaml")
cfg["data"]["root"] = NIH
cfg["data"]["image_root"] = IMAGE_ROOT
cfg["partition"]["manifest_path"] = f"{NIH}/manifest.json"

cfg["data"]["subsample"]["max_images"] = 30000   # raise for a bigger/longer run
cfg["training"]["rounds"] = 12                    # federated rounds
cfg["training"]["local_epochs"] = cfg["training"]["rounds"]   # fair ceiling
print("image_size:", cfg["data"]["image_size"],
      "| subsample:", cfg["data"]["subsample"]["max_images"],
      "| class_balanced:", cfg["training"]["class_balanced"],
      "| epochs/rounds:", cfg["training"]["local_epochs"], "/", cfg["training"]["rounds"])

## 5. Run the core suite

Centralised -> single-site -> federated, all on the same partition and test set. Prints
per-epoch / per-round progress. This is the long cell.

In [ ]:
from src.experiments.suite import run_core_suite

OUT = "/kaggle/working/results/colab"
summary = run_core_suite(
    cfg, out_dir=OUT, data_root=NIH, fed_local_epochs=1,
    extra_setup={"dataset": "NIH ChestX-ray14 (subset, class-balanced)",
                 "image_size": cfg["data"]["image_size"],
                 "baseline_epochs": cfg["training"]["local_epochs"]},
)

## 6. Charts + report

In [ ]:
from src.evaluation.report import generate_report
generate_report(OUT)

from IPython.display import Image, display, Markdown
for p in ["03_floor_fed_ceiling", "01_learning_curve", "02_global_loss", "04_per_pathology_auc"]:
    display(Image(f"{OUT}/plots/{p}.png"))
display(Markdown(open(f"{OUT}/REPORT.md").read()))

## 7. Download the results as a zip

Zips the results folder (JSON + plots + report) and shows a direct download link.
Click the link below to download `fl_results.zip`. (Everything under
`/kaggle/working/` is also saved as the notebook's Output if you commit the version.)

In [ ]:
import shutil
from IPython.display import FileLink, display
zip_base = "/kaggle/working/fl_results"
shutil.make_archive(zip_base, "zip", OUT)
print("zipped ->", zip_base + ".zip")
display(FileLink("fl_results.zip"))   # click to download